# Module 5 — Serving Local Models

Companion notebook to [`docs/modules/05_serving_local_models.md`](../docs/modules/05_serving_local_models.md).

This notebook:

1. Demonstrates the streaming NDJSON parser against fixture data shaped like real Ollama
   output (no server needed).
2. Demonstrates the model-metadata parser against a fixture `/api/show` response.
3. Renders the runtime feature comparison matrix.
4. Runs the real labs (streaming, metadata, warmup, OpenAI-compatible streaming, MLX) against
   live runtimes, if available — expected to skip on this machine (see `PROGRESS.md`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_05"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. Streaming NDJSON parsing, against fixture data

Real Ollama `/api/generate` streaming output, captured as fixture lines - no server needed
to prove the parser and TTFT/tokens-per-second math are correct.

In [2]:
import json

from ollama_streaming import accumulate_stream, iter_stream_chunks

fixture_lines = [
    json.dumps({"response": "The", "done": False}),
    json.dumps({"response": " quick", "done": False}),
    json.dumps({"response": " brown", "done": False}),
    json.dumps({"response": " fox", "done": False}),
    json.dumps({"response": "", "done": True, "eval_count": 4, "eval_duration": 400_000_000}),
]

chunks = list(iter_stream_chunks(fixture_lines))
# Simulate arrival timestamps: first chunk at 50ms (TTFT), then roughly evenly spaced.
start_time = 0.0
timestamps = [0.05, 0.15, 0.25, 0.35, 0.40]
result = accumulate_stream(zip(chunks, timestamps), start_time)

print(f"Full text: {result.full_text!r}")
print(f"TTFT: {result.ttft_seconds:.3f}s (real time-to-first-token, not the load+prompt-eval approximation Module 1 used)")
print(f"Total: {result.total_seconds:.3f}s")
print(f"Tokens/sec: {result.tokens_per_second:.2f}")

Full text: 'The quick brown fox'
TTFT: 0.050s (real time-to-first-token, not the load+prompt-eval approximation Module 1 used)
Total: 0.400s
Tokens/sec: 10.00


## 2. Model metadata parsing, against a fixture `/api/show` response

In [3]:
from ollama_metadata import parse_show_response, metadata_to_markdown
from IPython.display import Markdown, display

fixture_show_response = {
    "template": "{{ if .System }}<|im_start|>system...",
    "details": {
        "format": "gguf", "family": "qwen2", "parameter_size": "7.6B", "quantization_level": "Q4_K_M",
    },
    "model_info": {"qwen2.context_length": 32768},
}
metadata = parse_show_response("qwen2.5:7b", fixture_show_response)
display(Markdown(metadata_to_markdown(metadata)))

| Field | Value |
|---|---|
| model | qwen2.5:7b |
| family | qwen2 |
| parameter_size | 7.6B |
| quantization_level | Q4_K_M |
| format | gguf |
| context_length | 32768 |


## 3. Runtime feature comparison matrix

Every row is currently "documented" (from public runtime docs), not "measured" — see the
caveat in `feature_matrix.py`. This flips to "measured" per-feature once actually exercised
on a resourced Mac.

In [4]:
from feature_matrix import KNOWN_FEATURE_MATRIX, matrix_to_markdown, unverified_entries

display(Markdown(matrix_to_markdown(KNOWN_FEATURE_MATRIX)))
print(f"\n{len(unverified_entries(KNOWN_FEATURE_MATRIX))} of {len(KNOWN_FEATURE_MATRIX) * 6} feature entries still pending real measurement.")

| Runtime | Structured output | Grammar | Token counting | Streaming | Cancellation | Usage reporting |
|---|---|---|---|---|---|---|
| ollama | yes (documented) | no (documented) | partial (documented) | yes (documented) | yes (documented) | yes (documented) |
| llama.cpp (llama-server) | yes (documented) | yes (documented) | yes (documented) | yes (documented) | yes (documented) | yes (documented) |
| llama-cpp-python[server] | yes (documented) | yes (documented) | yes (documented) | yes (documented) | partial (documented) | yes (documented) |
| MLX / mlx-lm | no (documented) | no (documented) | partial (documented) | yes (documented) | n/a (documented) | partial (documented) |


24 of 24 feature entries still pending real measurement.


## 4. Warmup experiment, with a fake TTFT function (proves the statistics, no model needed)

In [5]:
from warmup_experiment import run_warmup_experiment, result_to_markdown

call_count = {"n": 0}

def fake_ttft(model, prompt):
    call_count["n"] += 1
    # First call (cold) is slow; subsequent (warm) calls are fast - simulating the real effect
    # this lab measures on a live runtime (theory doc §6).
    return 1.2 if call_count["n"] == 1 else 0.15

result = run_warmup_experiment("fake-model", "prompt", fake_ttft, n_warm_calls=3)
print(result_to_markdown(result))

# Warmup experiment — fake-model

- Cold TTFT: 1.200s
- Warm TTFTs: 0.150s, 0.150s, 0.150s
- Mean warm TTFT: 0.150s
- Speedup (cold / mean warm): 8.00x



## 5. Run the real labs against live runtimes, if available

Expected to skip on this machine (no local model runtime installed by design).

In [6]:
from ollama_probe import is_ollama_available

if is_ollama_available():
    from ollama_streaming import stream_generate
    real_result = stream_generate("qwen2.5:1.5b", "Say hello in five words.")
    print(f"Real TTFT: {real_result.ttft_seconds:.3f}s")
    print(f"Real response: {real_result.full_text}")
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On the resourced 32GB Mac: start Ollama (scripts/module_05/serve_ollama.sh), then re-run this cell.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On the resourced 32GB Mac: start Ollama (scripts/module_05/serve_ollama.sh), then re-run this cell.


In [7]:
import run_mlx_generate

if run_mlx_generate.is_apple_silicon() and run_mlx_generate.check_mlx_importable():
    summary = run_mlx_generate.run_warmup_and_streaming_demo(
        run_mlx_generate.DEFAULT_MODEL, run_mlx_generate.DEFAULT_PROMPT
    )
    print(run_mlx_generate.summary_to_markdown(summary))
else:
    print("SKIPPED: MLX is not available (not Apple Silicon, or mlx_lm not installed).")
    print("This machine deliberately has no model runtime installed (see PROGRESS.md).")

SKIPPED: MLX is not available (not Apple Silicon, or mlx_lm not installed).
This machine deliberately has no model runtime installed (see PROGRESS.md).


## 6. Take it to the command line

```bash
bash scripts/module_05/serve_ollama.sh qwen2.5:3b
uv run python scripts/module_05/ollama_metadata.py --model qwen2.5:3b
uv run python scripts/module_05/run_mlx_generate.py --model mlx-community/Qwen2.5-1.5B-Instruct-4bit

bash scripts/module_05/serve_llamacpp.sh python /path/to/model.gguf
uv run python scripts/module_05/llamacpp_openai_streaming.py
```

Then fold the real output into
[`reports/module_05_runtime_serving_matrix.md`](../reports/module_05_runtime_serving_matrix.md),
flipping the relevant `feature_matrix.py` entries from `verified=False` to `verified=True`
with real behavioral notes.